# Regresion Supervisada MP2.5 a 24 horas

Este notebook reutiliza `src/regression.py` para entrenar, evaluar y exportar el modelo sin duplicar logica.

In [1]:
import os
from pathlib import Path
import sys

def es_raiz_proyecto(path: Path) -> bool:
    return (
        (path / "docker-compose.yml").is_file()
        and (path / "src").is_dir()
        and (path / "notebooks").is_dir()
    )

def encontrar_raiz_proyecto() -> Path:
    candidatos = []

    for base in [Path.cwd(), *Path.cwd().parents]:
        candidatos.append(base)

    for env_key in ("__vsc_ipynb_file__", "JPY_SESSION_NAME", "PWD", "VSCODE_CWD"):
        valor = os.environ.get(env_key)
        if not valor:
            continue

        ruta = Path(valor).expanduser()
        if ruta.suffix == ".ipynb":
            ruta = ruta.parent

        candidatos.extend([ruta, *ruta.parents])

    revisados = set()
    for candidato in candidatos:
        candidato = candidato.resolve()
        if candidato in revisados:
            continue
        revisados.add(candidato)
        if es_raiz_proyecto(candidato):
            return candidato

    raise FileNotFoundError(
        "No se pudo ubicar la raiz del proyecto desde el notebook. "
        "Abre Jupyter desde la carpeta del repo o ejecuta el notebook dentro del workspace."
    )

PROJECT_ROOT = encontrar_raiz_proyecto()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.regression import ejecutar_pipeline_regresion, obtener_importancia_variables

print(f"PROJECT_ROOT: {PROJECT_ROOT}")

PROJECT_ROOT: E:\ProjectsGithub\air-quality-monitoring-platform


In [2]:
resultado = ejecutar_pipeline_regresion()
resultado.metricas

{'horizon_hours': 24,
 'sampling_frequency_hours': 6,
 'target_shift_steps': 4,
 'train_rows': 22,
 'test_rows': 6,
 'best_model': 'random_forest',
 'models': {'linear_regression': {'mae': 133.66883844398575,
   'rmse': 147.47138851399862,
   'r2': -169.03022531574558},
  'random_forest': {'mae': 8.374499999999996,
   'rmse': 10.024910064249756,
   'r2': 0.2142732083858192}}}

In [3]:
resultado.predicciones.head()

,tipo_registro,fecha_hora_base,fecha_hora_objetivo,fecha_hora,codigo_estacion,comuna,region,tipo_sensor,mp25_actual,mp25_real,mp25_real_24h,mp25_predicho,mp25_predicho_24h,error_absoluto,categoria_predicha,categoria_alerta_predicha,mensaje_ciudadano,color_referencial
0,evaluacion,2026-09-13 12:00:00,2026-09-14 12:00:00,2026-09-14 12:00:00,ONG-CON-001,Concepción,Biobío,sensor_comunitario_ong,22.6,24.8,24.8,43.094333,43.094333,18.294333,Buena,Buena,Calidad del aire favorable para actividades al...,verde
1,evaluacion,2026-09-13 18:00:00,2026-09-14 18:00:00,2026-09-14 18:00:00,ONG-CON-001,Concepción,Biobío,sensor_comunitario_ong,35.0,38.6,38.6,43.532667,43.532667,4.932667,Buena,Buena,Calidad del aire favorable para actividades al...,verde
2,evaluacion,2026-09-14 00:00:00,2026-09-15 00:00:00,2026-09-15 00:00:00,ONG-COR-001,Coronel,Biobío,sensor_comunitario_ong,52.9,55.0,55.0,50.877000,50.877000,4.123,Regular,Regular,Personas sensibles deberian reducir exposicion...,amarillo
3,evaluacion,2026-09-14 06:00:00,2026-09-15 06:00:00,2026-09-15 06:00:00,ONG-COR-001,Coronel,Biobío,sensor_comunitario_ong,49.5,51.7,51.7,46.203667,46.203667,5.496333,Buena,Buena,Calidad del aire favorable para actividades al...,verde
4,evaluacion,2026-09-14 12:00:00,2026-09-15 12:00:00,2026-09-15 12:00:00,ONG-COR-001,Coronel,Biobío,sensor_comunitario_ong,28.8,29.9,29.9,43.363667,43.363667,13.463667,Buena,Buena,Calidad del aire favorable para actividades al...,verde


In [4]:
obtener_importancia_variables(resultado.modelos[resultado.mejor_modelo].pipeline, top_n=10)

,feature,importance
0,mp25,0.459078
1,mp10,0.170815
2,humedad,0.065198
3,temperatura,0.061198
4,lag_1,0.049572
5,direccion_viento_cos,0.028267
6,direccion_viento_grados,0.020002
7,velocidad_viento,0.019713
8,so2,0.017750
9,direccion_viento_sin,0.016839
